# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We'll use the Croissant schema URL to instantiate the dataset, view metadata, and get a high-level summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print dataset summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out the dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and all corresponding `@id`s. All entities in this analysis are referenced by their Croissant `@id` fields for clear, unambiguous referencing.

In [ ]:
# List all record sets by their @id
print("Record sets (by @id):")
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"- @id: {rs.id}  |  name: {rs.name}")

# For each record set, list its fields and their @id
for rs in record_sets:
    print(f"\nFields for record set @id: {rs.id} ({rs.name}):")
    for field in rs.fields:
        col_info = ''
        if hasattr(field, 'column') and field.column:
            if isinstance(field.column, list):
                col_info = ', columns: ' + ', '.join([c.id for c in field.column if hasattr(c, 'id')])
            else:
                col_info = f", column: {field.column.id}"
        print(f"  - @id: {field.id}  |  name: {field.name}  |  dataType: {field.data_type}{col_info}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s you want to analyze from the overview above.

Here we demonstrate extracting and previewing each record set listed in the dataset.

In [ ]:
# Extract records from each record set and load into DataFrames
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# You may select a specific record set for further analysis below.
target_record_set_id = record_set_ids[0] if record_set_ids else None
if target_record_set_id:
    print(f"Selected record set for analysis: {target_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping/categorizing data. All field references use their Croissant `@id`.

*Example: We filter by a numeric field (e.g., 'Molecular weight (MW)') and normalize it, then group by another field, if available.*

In [ ]:
import numpy as np

# Example: Try to guess a numeric field @id from the previous overview. (Modify as appropriate for your dataset)
numeric_field_id = None
if target_record_set_id and target_record_set_id in dataframes:
    available_fields = dataframes[target_record_set_id].columns.tolist()
    # Try to auto-select a field likely to be numeric
    for candidate in available_fields:
        if 'weight' in candidate.lower() or 'mw' in candidate.lower():
            numeric_field_id = candidate
            break
    if not numeric_field_id:
        # Fallback to first column with numeric dtype
        for col in available_fields:
            if pd.api.types.is_numeric_dtype(dataframes[target_record_set_id][col]):
                numeric_field_id = col
                break

    print(f"Numeric field selected for analysis: {numeric_field_id}")

    # Proceed with filtering/analysis if field found
    if numeric_field_id and pd.api.types.is_numeric_dtype(dataframes[target_record_set_id][numeric_field_id]):
        threshold = np.nanpercentile(dataframes[target_record_set_id][numeric_field_id], 50)  # median as threshold
        filtered_df = dataframes[target_record_set_id][dataframes[target_record_set_id][numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by an available categorical field
        group_field_id = None
        for col in available_fields:
            if 'sample' in col.lower() or 'category' in col.lower() or (pd.api.types.is_object_dtype(dataframes[target_record_set_id][col]) and col != numeric_field_id):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field available for this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Examples include:

- Histogram of a numeric field (e.g., molecular weights)
- Boxplot of numeric field grouped by a category (if available)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram for the selected numeric field
if target_record_set_id and target_record_set_id in dataframes and numeric_field_id and pd.api.types.is_numeric_dtype(dataframes[target_record_set_id][numeric_field_id]):
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[target_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()

    # If there's a categorical field for grouping, plot boxplot
    if 'group_field_id' in locals() and group_field_id and group_field_id in dataframes[target_record_set_id].columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[target_record_set_id].dropna(subset=[group_field_id, numeric_field_id]))
        plt.xticks(rotation=45)
        plt.title(f"'{numeric_field_id}' Distribution by '{group_field_id}'")
        plt.show()

## 6. Conclusion
In this notebook, you learned how to load, inspect, preprocess, and visualize records from the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) Croissant dataset using the `mlcroissant` library. You referenced entities with their Croissant `@id`, explored available record sets and their fields, and performed basic EDA and visualization. For further biological or statistical analyses, please refer to the full field catalog and dataset description for additional details and recommendations.
